# 🏛️ Cargar los viajes de Ecobici en el lakehouse

Esta notebook hace la **ingesta**: baja los viajes en Ecobici de 2025, que publica la Ciudad de Buenos Aires en [Buenos Aires Data](https://data.buenosaires.gob.ar/dataset/bicicletas-publicas), y deja las tablas listas para consultar. Son más de 3 millones de viajes.

Los datos pasan por tres capas. Es la arquitectura medallón que vimos en la clase de data warehouse:

- 🥉 **Bronce**: los datos tal como llegan.
- 🥈 **Plata**: los datos limpios y con el tipo correcto.
- 🥇 **Oro**: las tablas listas para el análisis, armadas como un modelo estrella de hechos y dimensiones.

Las consultas no van acá: se hacen después, desde el editor SQL, sobre las tablas de oro.

> Para ejecutarla completa, usá el botón **Run all** de arriba. Tarda unos 3 minutos. También se puede ir celda por celda, con el botón ▶ de cada una o con Shift + Enter.

## 🗂️ Dónde se guardan los datos

Databricks organiza los datos en tres niveles: **catálogo**, **esquema** y **tabla**. El catálogo `workspace` ya viene creado. Adentro creamos el esquema `ecobici` y, dentro del esquema, un **volumen**: una carpeta para guardar archivos tal como vienen.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.ecobici;
CREATE VOLUME IF NOT EXISTS workspace.ecobici.archivos;

## 📥 Los datos

Bajamos al volumen el archivo de 2025: un ZIP de unos 200 MB con un CSV de más de 800 MB adentro. El servidor de la Ciudad entrega cada conexión despacio, así que bajamos el archivo en 16 partes a la vez, igual que con `aria2c` en la notebook de Parquet. Tarda alrededor de un minuto.

In [ ]:
import urllib.request
import zipfile
from concurrent.futures import ThreadPoolExecutor

URL = "https://cdn.buenosaires.gob.ar/datosabiertos/datasets/transporte-y-obras-publicas/bicicletas-publicas/recorridos-realizados-2025.zip"
VOLUMEN = "/Volumes/workspace/ecobici/archivos"
PARTES = 16

# Cuánto pesa el archivo
consulta = urllib.request.Request(URL, method="HEAD")
tamanio = int(urllib.request.urlopen(consulta).headers["Content-Length"])

# Cada parte baja un pedazo del archivo
def bajar_parte(numero):
    desde = numero * tamanio // PARTES
    hasta = (numero + 1) * tamanio // PARTES - 1
    pedido = urllib.request.Request(URL, headers={"Range": f"bytes={desde}-{hasta}"})
    return urllib.request.urlopen(pedido).read()

# Bajamos las 16 partes a la vez y las juntamos en un solo archivo
with ThreadPoolExecutor(PARTES) as grupo:
    partes = list(grupo.map(bajar_parte, range(PARTES)))

with open(f"{VOLUMEN}/recorridos-2025.zip", "wb") as archivo:
    for parte in partes:
        archivo.write(parte)

# Descomprimimos el ZIP en el volumen
zipfile.ZipFile(f"{VOLUMEN}/recorridos-2025.zip").extractall(VOLUMEN)
display(dbutils.fs.ls(VOLUMEN))

## 🥉 La capa de bronce

La tabla de bronce guarda los datos tal como llegan, sin cambios. Con `inferColumnTypes => false` le pedimos a Databricks que no adivine los tipos: todas las columnas quedan como texto, igual que en el archivo. Si algo sale mal en las capas siguientes, siempre se puede volver a empezar desde acá.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.ecobici.bronce_viajes AS
SELECT *
FROM read_files(
  '/Volumes/workspace/ecobici/archivos/Trips.csv',
  format => 'csv',
  header => true,
  inferColumnTypes => false
);

SELECT * FROM workspace.ecobici.bronce_viajes LIMIT 5;

Los datos tienen tres cosas para corregir: los identificadores terminan en `BAEcobici`, las duraciones de 1.000 segundos o más vienen con una coma (`1,016`) y el género está en inglés.

In [ ]:
%sql
SELECT Id_recorrido, duracion_recorrido, id_estacion_origen, `género`
FROM workspace.ecobici.bronce_viajes
WHERE duracion_recorrido LIKE '%,%'
LIMIT 5;

## 🥈 La capa de plata

En la tabla de plata corregimos cada columna:

- Sacamos el sufijo `BAEcobici` de los identificadores y los convertimos en números.
- Sacamos la coma de la duración.
- Convertimos las fechas y las coordenadas.
- Pasamos el género a castellano.
- Descartamos los viajes de más de un día, que son errores de registro.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.ecobici.plata_viajes AS
SELECT
  CAST(REPLACE(Id_recorrido, 'BAEcobici', '') AS BIGINT) AS id_viaje,
  CAST(REPLACE(id_usuario, 'BAEcobici', '') AS BIGINT) AS id_usuario,
  CAST(fecha_origen_recorrido AS TIMESTAMP) AS inicio,
  CAST(fecha_destino_recorrido AS TIMESTAMP) AS fin,
  CAST(REPLACE(duracion_recorrido, ',', '') AS INT) AS duracion_seg,
  CAST(REPLACE(id_estacion_origen, 'BAEcobici', '') AS INT) AS estacion_origen,
  nombre_estacion_origen,
  direccion_estacion_origen,
  CAST(lat_estacion_origen AS DOUBLE) AS lat_estacion_origen,
  CAST(long_estacion_origen AS DOUBLE) AS long_estacion_origen,
  CAST(REPLACE(id_estacion_destino, 'BAEcobici', '') AS INT) AS estacion_destino,
  nombre_estacion_destino,
  direccion_estacion_destino,
  CAST(lat_estacion_destino AS DOUBLE) AS lat_estacion_destino,
  CAST(long_estacion_destino AS DOUBLE) AS long_estacion_destino,
  modelo_bicicleta AS modelo,
  CASE `género`
    WHEN 'FEMALE' THEN 'Mujer'
    WHEN 'MALE' THEN 'Varón'
    WHEN 'OTHER' THEN 'Otro'
    ELSE 'Sin dato'
  END AS genero
FROM workspace.ecobici.bronce_viajes
WHERE CAST(REPLACE(duracion_recorrido, ',', '') AS INT) <= 24 * 60 * 60;

SELECT * FROM workspace.ecobici.plata_viajes LIMIT 5;

## 🥇 La capa de oro

En oro armamos un modelo estrella para el análisis: una tabla de hechos con una fila por viaje, rodeada de dimensiones que describen dónde y cuándo pasó cada viaje.

- `dim_estacion`: una fila por estación, con su nombre, su dirección y su ubicación.
- `dim_fecha`: una fila por día, con el mes, el día de la semana y si es fin de semana.
- `hechos_viajes`: una fila por viaje, con las claves de las dimensiones y la duración en minutos.

Las estaciones salen de los orígenes y de los destinos de los viajes, por eso juntamos las dos listas con `UNION ALL`.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.ecobici.hechos_viajes AS
SELECT
  id_viaje,
  DATE(inicio) AS fecha,
  HOUR(inicio) AS hora,
  estacion_origen,
  estacion_destino,
  modelo,
  genero,
  ROUND(duracion_seg / 60, 1) AS duracion_min
FROM workspace.ecobici.plata_viajes;

CREATE OR REPLACE TABLE workspace.ecobici.dim_estacion AS
SELECT
  id_estacion,
  MAX(nombre) AS nombre,
  MAX(direccion) AS direccion,
  MAX(latitud) AS latitud,
  MAX(longitud) AS longitud
FROM (
  SELECT estacion_origen AS id_estacion, nombre_estacion_origen AS nombre, direccion_estacion_origen AS direccion,
    lat_estacion_origen AS latitud, long_estacion_origen AS longitud
  FROM workspace.ecobici.plata_viajes
  UNION ALL
  SELECT estacion_destino, nombre_estacion_destino, direccion_estacion_destino,
    lat_estacion_destino, long_estacion_destino
  FROM workspace.ecobici.plata_viajes
)
WHERE id_estacion IS NOT NULL
GROUP BY id_estacion;

CREATE OR REPLACE TABLE workspace.ecobici.dim_fecha AS
SELECT DISTINCT
  fecha,
  MONTH(fecha) AS mes,
  WEEKDAY(fecha) + 1 AS numero_dia,
  ELEMENT_AT(ARRAY('lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado', 'domingo'), WEEKDAY(fecha) + 1) AS dia,
  WEEKDAY(fecha) >= 5 AS fin_de_semana
FROM workspace.ecobici.hechos_viajes;

SELECT * FROM workspace.ecobici.dim_fecha ORDER BY fecha LIMIT 7;

### Las claves

Declaramos la clave primaria de cada dimensión y las claves foráneas de la tabla de hechos. En Databricks estas claves son informativas: no impiden cargar datos que no las cumplan, pero documentan el modelo y permiten ver el diagrama de la estrella en el catálogo.

In [ ]:
%sql
ALTER TABLE workspace.ecobici.dim_estacion ALTER COLUMN id_estacion SET NOT NULL;
ALTER TABLE workspace.ecobici.dim_estacion ADD CONSTRAINT pk_estacion PRIMARY KEY (id_estacion);

ALTER TABLE workspace.ecobici.dim_fecha ALTER COLUMN fecha SET NOT NULL;
ALTER TABLE workspace.ecobici.dim_fecha ADD CONSTRAINT pk_fecha PRIMARY KEY (fecha);

ALTER TABLE workspace.ecobici.hechos_viajes ADD CONSTRAINT fk_fecha FOREIGN KEY (fecha) REFERENCES workspace.ecobici.dim_fecha;
ALTER TABLE workspace.ecobici.hechos_viajes ADD CONSTRAINT fk_origen FOREIGN KEY (estacion_origen) REFERENCES workspace.ecobici.dim_estacion;
ALTER TABLE workspace.ecobici.hechos_viajes ADD CONSTRAINT fk_destino FOREIGN KEY (estacion_destino) REFERENCES workspace.ecobici.dim_estacion;

## ✅ Las tablas quedaron listas

Una última consulta para ver cuántas filas tiene cada tabla. Plata tiene menos filas que bronce porque descartamos los viajes de más de un día.

In [ ]:
%sql
SELECT 'bronce_viajes' AS tabla, COUNT(*) AS filas FROM workspace.ecobici.bronce_viajes
UNION ALL SELECT 'plata_viajes', COUNT(*) FROM workspace.ecobici.plata_viajes
UNION ALL SELECT 'hechos_viajes', COUNT(*) FROM workspace.ecobici.hechos_viajes
UNION ALL SELECT 'dim_estacion', COUNT(*) FROM workspace.ecobici.dim_estacion
UNION ALL SELECT 'dim_fecha', COUNT(*) FROM workspace.ecobici.dim_fecha;

## 🧾 Para llevarse

- La ingesta se escribe una vez y se puede volver a ejecutar cuando lleguen datos nuevos. En una empresa, esta notebook se programa con **Jobs** para que corra sola, por ejemplo todas las noches.
- Cada capa es una tabla Delta: archivos Parquet más un registro de cambios. Por eso cada tabla tiene su historial de versiones.
- Las tablas quedan en el catálogo, en `workspace.ecobici`. Desde ahí las consulta cualquier herramienta: el editor SQL, un tablero o Genie.

**Siguiente paso:** abrir el **Catalog** para ver las tablas y después consultarlas desde el **SQL Editor**, como explica la guía.